In [1]:
# Libraries
import numpy as np
from qiskit.quantum_info import partial_trace
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import *
from qiskit_aer import StatevectorSimulator
import matplotlib.pyplot as plt
from matplotlib import animation
from PIL import Image
from ipywidgets import interact, widgets
from IPython.display import display, clear_output
import os
from itertools import chain

import warnings
import soundfile
warnings.filterwarnings('ignore')

backend = StatevectorSimulator(method="statevector")

In [2]:
# Functions

# qip/helper.py

def sfwht(a):
    """Fast walsh hadamard transform with scaling

    Args:
        a (flat array): array with values to be transformed

    Returns:
        input array: array of same type as input, inplace transform
    """
    n = len(a)
    k = ilog2(n)
    j = 1
    while j < n:
        for i in range(n):
            if i & j == 0:
                j1 = i + j
                x = a[i]
                y = a[j1]
                a[i], a[j1] = (x + y) / 2, (x - y) / 2
        j *= 2
    return a            

def isfwht(a):
    """Inverse of the walsh hadamard transform

    Args:
        a (array): array of values

    Returns:
        array: array with inverse transformed applied, inplace
    """
    n = len(a)
    k = ilog2(n)
    j=1
    while j< n:
        for i in range(n):
            if (i&j) == 0:
                j1=i+j 
                x=a[i]
                y=a[j1]
                a[i],a[j1]=(x+y),(x-y)
        j*=2 
    return a            

def ispow2(x):
    """am I a power of two

    Args:
        x (int): number

    Returns:
        Bool: is it a power of two? The answer
    """
    return not (x&x-1)

def nextpow2(x):
    """Returns next power of two, or identity if x is a power of two

    Args:
        x (int): number to check

    Returns:
        int: next power of two (or x if x is a power of two)
    """
    x-=1
    x|=x>>1
    x|=x>>2
    x|=x>>4
    x|=x>>8
    x|=x>>16
    x|=x>>32
    x+=1
    return x 

def ilog2(x):
    """Integer log 2"""
    return int(np.log2(x))

def grayCode(x):
    """Gray code permutation of x, to change indices"""
    return x^(x>>1)

def grayPermutation(a):
    """Gray permutes an array"""
    b = np.zeros(len(a))
    for i in range(len(a)):
        b[i] = a[grayCode(i)]
    return b

def invGrayPermutation(a):
    """inverse gray permutes an array"""
    b = np.zeros(len(a))
    for i in range(len(a)):
        b[grayCode(i)] = a[i]
    return b

def convertToAngles(a):
    """Converts image to angles"""
    scal = np.pi/(a.max()*2)
    a = a *scal
    return a

def convertFromAngles(a,maxval=1,minval=0):
    """Converts image from angles"""
    scal = np.pi/(maxval*2)
    a = a /scal
    return a


def convertToGrayscaleOld(arr,maxval=1,minval=0):
    """Converts encoded postprocessed statevector back to grayscale, normalized to maxval"""
    scal = 2*(maxval)/np.pi
    arr = arr * scal
    # arr = ((arr - arr.min()+minval) * (1/(arr.max() - arr.min()) * maxval))
    return arr

def convertToGrayscale(arr, maxval=1, minval=0):
    """
    Scales an array so that its minimum and maximum values lie between new_min and new_max.

    Args:
        arr (numpy.ndarray): Input array to be scaled.
        new_min (float): The desired minimum value of the scaled array.
        new_max (float): The desired maximum value of the scaled array.

    Returns:
        numpy.ndarray: Scaled array with values between new_min and new_max.
    """
    old_min = np.min(arr)
    old_max = np.max(arr)
    scaled_arr = (arr - old_min) / (old_max - old_min) * (maxval - minval) + minval
    return scaled_arr

def countr_zero(n,n_bits=8):
    """Returns the number of consecutive 0 bits 
    in the value of x, starting from the 
    least significant bit ("right")."""
    if n == 0:
        return n_bits
    count = 0
    while n & 1 == 0:
        count += 1
        n >>= 1
    return count

def preprocess_image(img):
    """Program requires flattened transpose of image array, this returns exactly that"""
    return img.T.flatten()

def readpgm(name):
    """Reads pgm P2 files"""
    with open(name) as f:
        lines = f.readlines()
    # This ignores commented lines
    for l in list(lines):
        if l[0] == '#':
            lines.remove(l)
    # here,it makes sure it is ASCII format (P2)
    assert lines[0].strip() == 'P2' 
    # Converts data to a list of integers
    data = []
    for line in lines[1:]:
        data.extend([int(c) for c in line.split()])
        
    return (np.array(data[3:]),(data[1],data[0]),data[2])

def pad_0(img,val=0):
    """Pads array with 0s to next power of two

    Args:
        img (numpy array): image, can be wide

    Returns:
        padded image: flattened image with appropiate padding for quantum algorithm
    """
    img = np.array(img)
    img.flatten()
    return np.pad(img,(0,nextpow2(len(img))-len(img)),constant_values=val)

def decodeQPIXL(state,min_pixel_val=0,max_pixel_val=255, state_to_prob = np.abs, scaling = convertToGrayscale):
    """Automatically decodes qpixl output statevector

    Args:
        state (statevector array): statevector from simulator - beware of bit ordering
        max_pixel_val (int, optional): normalization value. Defaults to 255.
        state_to_prob (function): If you made some transforms, your image 
                                    may be complex, how would you 
                                    like to make the vector real?
    Returns:
        np.array: your image, flat
    """
    state_to_prob(state)
    pv = np.zeros(len(state)//2)
    for i in range(0,len(state),2):
        pv[i//2]=np.arctan2(state[i+1],state[i])
    return scaling(pv,max_pixel_val,min_pixel_val)

def permute_bits(b,bitlength=8,shift=1):
    """cyclic permutation of bits

    Args:
        b (integer): integer to be converted
        bitlength (int, optional): how many bits do you want to permute in. Defaults to 8.
        shift (int, optional): how many bits to shift. Defaults to 1.

    Returns:
        int: integer representation of bits
    """
    b = bin(b)
    b = b[2:].zfill(bitlength)
    b = [b[(i + shift) % len(b)] for i in range(len(b))]
    return int(''.join(b),2)

def decodeParallelQPIXL(state, qc, length ,normalization_values=None):
    """Automatically decodes qpixl output statevector

    Args:
        state (statevector array): statevector from simulator - beware of bit ordering
        qc (qiskit circuit): the circuit used for the state generation
        max_pixel_val (list of tuples of ints [(min,max)], optional): normalization values, must be of same length as length. Defaults to [(0,255)]*length.
    Returns:
        np.array: your images, flat
    """
    if normalization_values is None:
        normalization_values = [(0,255) for i in range(length)]
    decoded_data = []
    for datum in range(length):
        to_trace = list(range(length))
        popped = to_trace.pop(length-datum-1)
        to_trace = [qc.qubits[qub] for qub in to_trace]
        traced_over_qubits = [qc.qubits.index(qubit) for qubit in to_trace]
        density_matrix = partial_trace(state, traced_over_qubits)
        probs = density_matrix.probabilities()
        test = decodeQPIXL(probs)
        ordered = [test[permute_bits(i,len(qc.qubits)-length,datum)] for i in range(len(test))]
        decoded_data.append(convertToGrayscale(np.array(ordered),normalization_values[datum][1],normalization_values[datum][0]))
    return decoded_data


def reconstruct_img(pic_vec, shape: tuple):
    """reconstruct image from decoded statevector

    Args:
        pic_vec (np.array): your decoded statevector
        shape (tuple): shape that you want the image back in

    Returns:
        np.array: array of correct image size, ready to show! May need to be transposed.
    """
    ldm = shape[0]
    holder = np.zeros(shape)
    for row in range(shape[0]):
        for col in range(shape[1]):
            holder[row,col]=pic_vec[row + col * ldm]
    return holder

class examples():
    def __init__(self) -> None:
        """SImple holder class with some example images
        """
        self.space= np.array([  [0,0,0,0,1,1,1,0],
                                [0,0,0,1,1,0,0,0],
                                [1,0,1,1,1,1,1,0],
                                [0,1,1,0,1,1,0,1],
                                [0,0,1,1,1,1,0,1],
                                [0,0,1,1,1,1,0,0],
                                [0,0,1,1,1,1,0,1],
                                [0,1,1,0,1,1,0,1],
                                [1,0,1,1,1,1,1,0],
                                [0,0,0,1,1,0,0,0],
                                [0,0,0,0,1,1,1,0],
                                [0,0,0,0,0,0,0,0],
                                [0,0,0,0,0,0,0,0],
                                [0,0,0,0,0,0,0,0],
                                [0,0,0,0,0,0,0,0],
                                [0,0,0,0,0,0,0,0]])
        self.invader = np.array([[0,0,0,0,1,1,1,1],
                                 [0,1,1,1,1,1,0,0],
                                 [0,1,0,0,1,1,1,1],
                                 [0,1,0,1,1,1,0,0],
                                 [1,1,1,1,1,1,1,1],
                                 [1,1,1,1,1,1,0,0],
                                 [1,1,0,0,1,1,1,1],
                                 [0,1,0,1,1,1,0,0],
                                 [0,1,1,1,1,1,1,1],
                                 [0,1,1,1,1,1,0,0],
                                 [0,0,0,0,1,1,1,1],
                                 [0,0,0,0,0,0,0,0],
                                 [0,0,0,0,0,0,0,0],
                                 [0,0,0,0,0,0,0,0],
                                 [0,0,0,0,0,0,0,0],
                                 [0,0,0,0,0,0,0,0]])

In [3]:
def cFRQI_demo(a, compression):
    """    Takes a standard image in a numpy array (so that the matrix looks like
    the image you want if you picture the pixels) and returns the QPIXL
    compressed FRQI circuit. The compression ratio determines
    how many gates will be filtered and then cancelled out. Made into code from this paper:
    https://www.nature.com/articles/s41598-022-11024-y

    Args:
        a (np.array): numpy array of image, must be flattened and padded with zeros up to a power of two
        compression (float): number between 0 an 100, where 0 is no compression and 100 is no image

    Returns:
        QuantumCircuit: qiskit circuit that prepared the encoded image
    """
    a = convertToAngles(a) # convert grayscale to angles
    a = preprocess_image(a) # need to flatten the transpose for easier decoding, 
                            # only really necessary if you want to recover an image.
                            # for classification tasks etc. transpose isn't required.
    n = len(a)
    k = ilog2(n)
    a = 2*a 
    a = sfwht(a)
    a = grayPermutation(a) 
    a_sort_ind = np.argsort(np.abs(a))
    # set smallest absolute values of a to zero according to compression param
    cutoff = int((compression / 100.0) * n)
    for it in a_sort_ind[:cutoff]:
        a[it] = 0
    
    # Construct FRQI circuit
    dataqbits = qiskit.QuantumRegister(k,'storage qubits')
    encodingqubit = qiskit.QuantumRegister(1,'encoding qubit')
    circuit = QuantumCircuit(dataqbits, encodingqubit)
    
    # Data qubits
    circuit.h(dataqbits)

    ctrl, pc, i = 0, 0, 0
    while i < (2**k):
        pc = int(0) # Reset the parity check
        if a[i] != 0:
            circuit.ry(a[i],encodingqubit)  #normally would just be an ry gate 
        # Loop over sequence of consecutive zero angles to cancel out CNOTS (or rather, to not include them)
        if i == ((2**k) - 1):
            ctrl=0
        else:
            ctrl = grayCode(i) ^ grayCode(i+1)
            ctrl = k - countr_zero(ctrl, n_bits=k+1) - 1
        pc ^= (2**ctrl) # Update parity check
        i += 1
        while i < (2**k) and a[i] == 0:
            # Compute control qubit
            if i == ((2**k) - 1):
                ctrl=0
            else:
                ctrl = grayCode(i) ^ grayCode(i+1)
                ctrl = k - countr_zero(ctrl, n_bits=k+1) - 1
            pc ^= (2**ctrl) # Update parity check
            i += 1              
        for j in range(k):
            if (pc >> j)  &  1:
                circuit.cx(dataqbits[j], encodingqubit[0])      
    circuit.reverse_bits()
    return circuit
circ = cFRQI_demo(np.array([0,1,2,3,4,5,6,7]), 0)
circ.draw(fold=150,vertical_compression='high')

┌───┐                                                                       
storage qubits_0: ───┤ H ├─────────────────────────────────────────────■──────────────────────■──
                     ├───┤                                             │                      │  
storage qubits_1: ───┤ H ├───────────────────────■─────────────────────┼────■─────────────────┼──
                     ├───┤                       │                     │    │                 │  
storage qubits_2: ───┤ H ├─────■─────────────────┼────■────────────────┼────┼─────────────────┼──
                  ┌──┴───┴──┐┌─┴─┐┌───────────┐┌─┴─┐┌─┴─┐┌──────────┐┌─┴─┐┌─┴─┐┌───────────┐┌─┴─┐
  encoding qubit: ┤ Ry(π/2) ├┤ X ├┤ Ry(-π/14) ├┤ X ├┤ X ├┤ Ry(-π/7) ├┤ X ├┤ X ├┤ Ry(-2π/7) ├┤ X ├
                  └─────────┘└───┘└───────────┘└───┘└───┘└──────────┘└───┘└───┘└───────────┘└───┘

In [4]:
def cFRQIangs(a, compression, pre_pattern=None,post_pattern=None):
    """
    VARIANT FUNCTION - more of an exampole.
    Constructs the QPIXL circuit.
    This function takes an input image represented as a grayscale array, compresses it
    based on the specified compression ratio, and generates a quantum circuit that 
    encodes the image. The circuit can optionally include pre- and post-pattern operations around controlled rotations.
    Args:
        a (numpy.ndarray): Grayscale image represented as a 1D array of pixel intensities.
        compression (float): Compression ratio as a percentage (0-100). Determines the 
                                proportion of smallest absolute values in the image data 
                                to set to zero.
        pre_pattern (callable, optional): A function that applies a custom operation 
                                            to the circuit before a controlled rotation. 
                                            Defaults to None.
        post_pattern (callable, optional): A function that applies a custom operation 
                                            to the circuit after a controlled rotation. 
                                            Defaults to None.
    Returns:
        QuantumCircuit: A quantum circuit implementing the compressed FRQI representation 
                        of the input image.
    Notes:
        - The input array is first converted to angles and preprocessed.
        - The Walsh-Hadamard Transform (WHT) is applied to the data, followed by 
            a Gray code permutation.
        - The smallest absolute values in the transformed data are set to zero based on 
            the compression parameter.
        """
    a = convertToAngles(a) # convert grayscale to angles
    a = preprocess_image(a) # need to flatten the transpose for easier decoding, 
                            # only really necessary if you want to recover an image.
                            # for classification tasks etc. transpose isn't required.
    n = len(a)
    k = ilog2(n)

    a = 2*a 
    a = sfwht(a)
    a = grayPermutation(a) 
    a_sort_ind = np.argsort(np.abs(a))

    # set smallest absolute values of a to zero according to compression param
    cutoff = int((compression / 100.0) * n)
    for it in a_sort_ind[:cutoff]:
        a[it] = 0
    # print(a)
    # Construct FRQI circuit
    circuit = QuantumCircuit(k + 2)
    # Hadamard register
    circuit.h(range(2,k+2))
    circuit.x(0)
    # Compressed uniformly controlled rotation register
    ctrl, pc, i = 0, 0, 0
    while i < (2**k):
        # Reset the parity check
        pc = int(0)

        # Add RY gate
        if a[i] != 0:
            if pre_pattern is None:
                circuit.ry(a[i],1)
            else:
                pre_pattern(circuit)
                circuit.cry(a[i],0,1)
                post_pattern(circuit)
            
        # Loop over sequence of consecutive zero angles to 
        # cancel out CNOTS (or rather, to not include them)
        if i == ((2**k) - 1):
            ctrl=0
        else:
            ctrl = grayCode(i) ^ grayCode(i+1)
            ctrl = k - countr_zero(ctrl, n_bits=k+1) - 1

        # Update parity check
        pc ^= (2**ctrl)
        i += 1
        
        while i < (2**k) and a[i] == 0:
            # Compute control qubit
            if i == ((2**k) - 1):
                ctrl=0
            else:
                ctrl = grayCode(i) ^ grayCode(i+1)
                ctrl = k - countr_zero(ctrl, n_bits=k+1) - 1

            # Update parity check
            pc ^= (2**ctrl)
            i += 1
                        
        for j in range(k):
            if (pc >> j)  &  1:
                circuit.cx(k-j+1, 1)
    return circuit

def decodeAngQPIXL(state, qc, trace ,max_pixel_val=255, min_pixel_val=0):
    """Automatically decodes qpixl output statevector - taking into account 
    that there are other qubits in the cirucit - this version only returns one image. 

    Args:
        state (statevector array): statevector from simulator - beware of bit ordering
        qc (qiskit circuit): the circuit used for the state generation
        max/min_pixel_val (int, optional): normalization values. Defaults to 255/0.
    Returns:
        np.array: your image, flat
    """
    decoded_data = []
    datum=0
    to_trace = list(range(trace))
    to_trace.pop(trace-datum-1)
    test = decodeQPIXL(partial_trace(state, [qc.qubits.index(qubit) for qubit in [qc.qubits[qub] for qub in to_trace]]).probabilities())
    return convertToGrayscale(np.array([test[permute_bits(i,len(qc.qubits)-trace,datum)] for i in range(len(test))]),max_pixel_val,min_pixel_val)


In [5]:
### An example function for processing audio adding some interleaving gates (pattern functions) and some gates after the encoding (post processing function)

from qiskit import transpile


def process_audio(
    input_file, 
    output_dir, 
    pattern_func, 
    pattern2_func, 
    post_process_func=None, 
    tag="", 
    compression=0,
):
    """
    Processes an audio file by encoding it into a quantum circuit, applying transformations, 
    and performing spectral noise reduction. 

    Args:
        input_file (str): Path to the input audio file.
        output_dir (str): Directory to save the processed audio files.
        pattern_func (function): Function to apply the first pattern to the quantum circuit.
        pattern2_func (function): Function to apply the second pattern to the quantum circuit.
        post_process_func (function, optional): Function to apply additional processing to the quantum circuit. Defaults to None.
        tag (str): Tag to append to the output file names. Defaults to an empty string.
        compression (int): Compression level for the quantum circuit encoding. Defaults to 0.

    Returns:
        None
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    file_name, _ = os.path.splitext(os.path.basename(input_file))
    metadata_file_path = os.path.join(output_dir, f"{file_name}_metadata_{tag}_c{compression}.txt")
    if not os.path.exists(metadata_file_path):
    # Read the input audio file and split, as it would be too large to process in one go
        data, samplerate = soundfile.read(input_file)
        chunk_size = 512
        sections = [pad_0(data[i:i + chunk_size]) for i in range(0, len(data), chunk_size)][:-1] 
        #remove last chunk since it can cause issues due to 0s in padding
        decoded = []

        # Process each audio chunk
        for index, section in enumerate(sections):
            print(f'Processing {file_name} chunk: {index + 1}/{len(sections)}', end='\r')
            normalized_section = section - np.min(section)
            patt1,_ = pattern_func
            patt2,_ = pattern2_func
            qc = cFRQIangs(normalized_section, compression, patt1, patt2) # 
            ### Variant of encoding that has additional code to allow for a 
            ### pattern to be applied in the algorithm qubits

            # Apply post-processing if provided
            if post_process_func is not None:
                post_process_func(qc)

            # Run the quantum circuit and decode the result
            transpiled_qc = transpile(
                qc,
                backend=backend,
                optimization_level=1,
            )

            job = backend.run(transpiled_qc)
            state_vector = np.real(
                np.asarray(
                    job.result().get_statevector(transpiled_qc)
                )
            )
            decoded.append(decodeAngQPIXL(
                state=state_vector, 
                qc=qc, 
                trace=2, 
                max_pixel_val=section.max(), 
                min_pixel_val=section.min() #Ensures that the audio amplitudes are preserved between chunks
            ))

        # Save the processed audio
        decoded_full = np.array(list(chain.from_iterable(decoded)))
        original_with_pad = np.array(list(chain.from_iterable(sections)))
        soundfile.write(
            os.path.join(output_dir, f"{file_name}_output_{tag}_c{compression}.wav"), 
            decoded_full, 
            samplerate, 
        )
        
def pattern(angle):
    """    Example pattern function that applies a 
    CRX gate to the quantum circuit in the algortihm register.
    Returns a function that takes a circuit and applies the pattern.
    """
    def pattern(circ):
        circ.crx(angle, 1, 0)
    return pattern, angle
def pattern2(angle):
    def pattern(circ):
        circ.cry(angle, 1, 0)
    return pattern, angle 
def post(qubit):
    def post(circ):
        circ.h(qubit)
    return post

wav_files = [f for f in os.listdir('Sample_Material') if f.endswith('.mp3')]
folder_paths = [os.path.join('Sample_Material', os.path.splitext(f)[0]) for f in wav_files]
for file,path in zip(wav_files,folder_paths):
    process_audio(
                os.path.join('Sample_Material', file), 
                path, 
                pattern(np.pi/5), 
                pattern2(np.pi/10),
                post_process_func=post(3),
                compression=90,
                tag="demo", 
                )

AttributeError: 'Target' object has no attribute '_qarg_gate_map'